# Notebook 09 â€” Sequence Windowing and Label Preparation

**Part 4** of the Parkinson_Gait_CIGTNet preprocessing pipeline.

## Methodology

This notebook performs **fixed-length 751-frame window extraction** from each
eligible Part 3 normalized-and-scaled trial sequence.

- **Window length W = 751**: directly supported by the historical report's fixed
  normalized/model sequence length (Section 3.4.1: $X \in \mathbb{R}^{N \times T \times F}$, T=751).
- **Stride S = 751**: a derived implementation setting chosen to produce one
  complete non-overlapping window from each fixed-length normalized trial.
  The historical report did **not** specify a separate sliding-window stride.

Since every eligible trial is exactly 751 frames after GSN/TBSN normalization,
this configuration produces exactly **one window per trial** with no incomplete
trailing candidates.

**Safety**: This notebook performs **no automatic deletion** of existing data.
Raw and intermediate directories are strictly read-only.


In [1]:
from pathlib import Path
import ast
import inspect
import importlib
import sys

def find_project_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'configs' / 'data_config.yaml').exists():
            return candidate
    raise RuntimeError('Project root not found.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 1. Direct on-disk source check via AST
WINDOWING_PATH = PROJECT_ROOT / 'src' / 'preprocessing' / 'windowing.py'
source = WINDOWING_PATH.read_text(encoding='utf-8')
tree = ast.parse(source)
function_names = {
    node.name
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}
print('On-disk function names:', function_names)
if 'extract_sliding_windows' not in function_names:
    raise RuntimeError(
        'extract_sliding_windows is still missing from the actual on-disk windowing.py'
    )
print('PASS: extract_sliding_windows exists in the on-disk source file.')

# 2. Package import verification
import src.preprocessing.windowing as win
win = importlib.reload(win)

loaded_path = Path(win.__file__).resolve()
expected_path = WINDOWING_PATH.resolve()
print('Loaded windowing module:', loaded_path)
if loaded_path != expected_path:
    raise RuntimeError(f'FAIL: Loaded module from {loaded_path} instead of {expected_path}')

print('Has extract_sliding_windows:', hasattr(win, 'extract_sliding_windows'))
if not hasattr(win, 'extract_sliding_windows'):
    raise RuntimeError(
        'FAIL: extract_sliding_windows is missing from the imported module.'
    )

print('Signature:')
print(inspect.signature(win.extract_sliding_windows))
print('\nSource:')
print(inspect.getsource(win.extract_sliding_windows))


On-disk function names: {'extract_sliding_windows', 'extract_windows_from_trial', 'build_window_manifest'}
PASS: extract_sliding_windows exists in the on-disk source file.
Loaded windowing module: C:\Users\lab\Minor_project\Parkinson_gait\src\preprocessing\windowing.py
Has extract_sliding_windows: True
Signature:
(scaled_df: pandas.DataFrame, mask_df: pandas.DataFrame, feature_names: List[str], window_length: int, stride: int, trial_metadata: Dict[str, Any]) -> Tuple[List[numpy.ndarray], List[numpy.ndarray], List[Dict[str, Any]], List[Dict[str, Any]]]

Source:
def extract_sliding_windows(
    scaled_df: pd.DataFrame,
    mask_df: pd.DataFrame,
    feature_names: List[str],
    window_length: int,
    stride: int,
    trial_metadata: Dict[str, Any],
) -> Tuple[List[np.ndarray], List[np.ndarray], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Extract deterministic sliding windows from a single trial sequence.

    For a sequence of T frames, generates windows at positions
    [0, st

In [2]:
import os, json, time, yaml
import numpy as np
import pandas as pd
# import matplotlib
# matplotlib.use('Agg')
# import matplotlib.pyplot as plt

METADATA_DIR   = PROJECT_ROOT / 'data' / 'metadata'
SCALED_DIR     = PROJECT_ROOT / 'data' / 'processed' / 'scaled_features'
NORMALIZED_DIR = PROJECT_ROOT / 'data' / 'normalized'
FINAL_DIR      = PROJECT_ROOT / 'data' / 'final'
RAW_DIR        = PROJECT_ROOT / 'data' / 'raw'
RESULTS_TABLES = PROJECT_ROOT / 'results' / 'tables'
RESULTS_FIGS   = PROJECT_ROOT / 'results' / 'figures' / 'part4'
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_FIGS.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')


Project root: c:\Users\lab\Minor_project\Parkinson_gait


In [ ]:

with open(PROJECT_ROOT / 'configs' / 'data_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

W = config['windowing'].get('window_size')
S = config['windowing'].get('stride')

if W is None or S is None:
    raise RuntimeError(
        'PART_4_BLOCKED: window_size and/or stride are null in '
        'configs/data_config.yaml. Cannot proceed without authoritative values.'
    )

W = int(W)
S = int(S)
print(f'Window length W = {W}  (report-supported fixed sequence length)')
print(f'Stride        S = {S}  (derived implementation setting)')


Window length W = 751  (report-supported fixed sequence length)
Stride        S = 751  (derived implementation setting)


## Authoritative 70/15/15 Subject Split

Generate the report-defined split (70% Train, 15% Val, 15% Test) with subject-level separation
and stratification by medication state.


In [4]:
import numpy as np
import pandas as pd
np.random.seed(42)  # Fixed random seed for reproducibility

manifest_df = pd.read_csv(METADATA_DIR / 'merged_trial_manifest.csv')
eligible_df = manifest_df[manifest_df['merge_status'] == 'SUCCESS'].copy()

# Identify unique subjects and their medication states
subject_stats = eligible_df.groupby('subject_id').agg(
    trials=('trial_key', 'count'),
    on_trials=('condition', lambda x: (x == 'on').sum()),
    off_trials=('condition', lambda x: (x == 'off').sum())
).reset_index()

def get_med_state(row):
    if row['on_trials'] > 0 and row['off_trials'] > 0: return 'both'
    elif row['on_trials'] > 0: return 'on_only'
    elif row['off_trials'] > 0: return 'off_only'
    else: return 'none'

subject_stats['med_state'] = subject_stats.apply(get_med_state, axis=1)
subject_stats = subject_stats.sort_values('subject_id')

splits = {'TRAIN': [], 'VALIDATION': [], 'TEST': []}
for state, group in subject_stats.groupby('med_state'):
    subjects = group['subject_id'].tolist()
    np.random.shuffle(subjects)
    n = len(subjects)
    n_train = int(round(n * 0.70))
    n_val = int(round(n * 0.15))
    
    if n == 1:
        splits['TRAIN'].extend(subjects)
    elif n == 3:
        splits['TRAIN'].extend(subjects[:2])
        splits['VALIDATION'].extend(subjects[2:3])
    else:
        splits['TRAIN'].extend(subjects[:n_train])
        splits['VALIDATION'].extend(subjects[n_train:n_train+n_val])
        splits['TEST'].extend(subjects[n_train+n_val:])

train_subjects = set(splits['TRAIN'])
val_subjects = set(splits['VALIDATION'])
test_subjects = set(splits['TEST'])
all_eligible_subjects = set(subject_stats['subject_id'])

# 8. STRONG LEAKAGE ASSERTIONS
assert train_subjects.intersection(val_subjects) == set(), 'Leakage: TRAIN and VAL overlap!'
assert train_subjects.intersection(test_subjects) == set(), 'Leakage: TRAIN and TEST overlap!'
assert val_subjects.intersection(test_subjects) == set(), 'Leakage: VAL and TEST overlap!'
assert train_subjects.union(val_subjects).union(test_subjects) == all_eligible_subjects, 'Missing subjects from split!'

def get_split_name(sub):
    if sub in train_subjects: return 'TRAIN'
    if sub in val_subjects: return 'VALIDATION'
    if sub in test_subjects: return 'TEST'

subject_stats['split'] = subject_stats['subject_id'].apply(get_split_name)

subject_stats.rename(columns={
    'med_state': 'medication_states_present',
    'trials': 'number_of_trials',
    'on_trials': 'number_of_ON_trials',
    'off_trials': 'number_of_OFF_trials'
}, inplace=True)

if (METADATA_DIR / 'train_val_test_split.csv').exists():
    old_sp = pd.read_csv(METADATA_DIR / 'train_val_test_split.csv')
    assert set(old_sp[old_sp['split'] == 'TEST']['subject_id']) == test_subjects, 'TEST set mismatch (trial_split changed)'

subject_stats.to_csv(METADATA_DIR / 'train_val_test_split.csv', index=False)
pd.Series(list(train_subjects), name='subject_id').to_csv(METADATA_DIR / 'train_subjects.csv', index=False)
pd.Series(list(val_subjects), name='subject_id').to_csv(METADATA_DIR / 'validation_subjects.csv', index=False)
pd.Series(list(test_subjects), name='subject_id').to_csv(METADATA_DIR / 'test_subjects.csv', index=False)

# 7. SPLIT AUDIT
total_subjects = len(all_eligible_subjects)
total_trials = subject_stats['number_of_trials'].sum()
audit = []
for split in ['TRAIN', 'VALIDATION', 'TEST']:
    split_df = subject_stats[subject_stats['split'] == split]
    s_count = len(split_df)
    t_count = split_df['number_of_trials'].sum()
    audit.append({
        'Split': split,
        'Subjects': s_count,
        '% Subjects': round(100.0 * s_count / total_subjects, 1) if total_subjects else 0,
        'Trials': t_count,
        '% Trials': round(100.0 * t_count / total_trials, 1) if total_trials else 0,
        'ON Trials': split_df['number_of_ON_trials'].sum(),
        'OFF Trials': split_df['number_of_OFF_trials'].sum()
    })

audit_df = pd.DataFrame(audit)
print('--- SPLIT AUDIT ---')
print(audit_df.to_string(index=False))
print(f'\nTotal Subjects: {total_subjects} | Total Trials: {total_trials}')



--- SPLIT AUDIT ---
     Split  Subjects  % Subjects  Trials  % Trials  ON Trials  OFF Trials
     TRAIN        18        69.2     589      66.9        308         281
VALIDATION         4        15.4     134      15.2         81          53
      TEST         4        15.4     158      17.9         81          77

Total Subjects: 26 | Total Trials: 881


In [5]:
#  Load authoritative Part 3 artifacts 
feature_prov = pd.read_csv(METADATA_DIR / 'merged_feature_provenance.csv')
feature_names = feature_prov[
    feature_prov['source_modality'].isin({'linear_kinematics', 'angular_kinematics'})
]['merged_feature_name'].tolist()
F = len(feature_names)
print(f'Feature count F = {F}')

try:
    with open(METADATA_DIR / 'gsn_parameters.json', 'r') as f:
        gsn_params = json.load(f)
    gsn_target = int(gsn_params['target_length'])
    assert gsn_target == W, f'GSN target_length={gsn_target} != W={W}'
except FileNotFoundError:
    print('Warning: gsn_parameters.json not found, assuming W matches')

try:
    gsn_manifest_df = pd.read_csv(METADATA_DIR / 'gsn_manifest.csv')
    eligible_trial_keys = sorted(gsn_manifest_df['trial_key'].tolist())
except FileNotFoundError:
    eligible_trial_keys = sorted(manifest_df['trial_key'].tolist())

BRANCHES = ['gsn_minmax', 'gsn_zscore', 'tbsn_minmax', 'tbsn_zscore']
print(f'Eligible trials : {len(eligible_trial_keys)}')
print(f'Branches        : {BRANCHES}')

# Preserve 5-fold CV as a separate methodological experiment if needed, 
# but we establish trial_split as authoritative.
folds = [{'fold_id': 'trial_split', 'train_subjects': list(splits['TRAIN']), 'val_subjects': list(splits['VALIDATION']), 'test_subjects': list(splits['TEST'])}]



Feature count F = 180
Eligible trials : 881
Branches        : ['gsn_minmax', 'gsn_zscore', 'tbsn_minmax', 'tbsn_zscore']


In [ ]:

clinical_raw = pd.read_excel(RAW_DIR / 'PDGinfo.xlsx')

# Exact column names (note trailing space on Hoehn & Yahr columns)
COL_ID           = 'ID'
COL_ON_UPDRS_WK  = 'ON - UPDRS-III - walking'
COL_OFF_UPDRS_WK = 'OFF - UPDRS-III - walking'
COL_ON_HY        = 'ON - Hoehn & Yahr '   # trailing space
COL_OFF_HY       = 'OFF - Hoehn & Yahr '  # trailing space
COL_ON_UPDRS3    = 'ON - UPDRS-III'
COL_OFF_UPDRS3   = 'OFF - UPDRS-III'

for col in [COL_ID, COL_ON_UPDRS_WK, COL_OFF_UPDRS_WK,
            COL_ON_HY, COL_OFF_HY, COL_ON_UPDRS3, COL_OFF_UPDRS3]:
    assert col in clinical_raw.columns, f'Missing column: {repr(col)}'

clinical = clinical_raw.copy()
clinical.replace('-', np.nan, inplace=True)
clinical_subjects = sorted(clinical[COL_ID].dropna().tolist())
assert len(clinical_subjects) == 26

# Build subject-level label DataFrame (one row per subject)
label_records = []
for _, row in clinical.iterrows():
    sub = row[COL_ID]
    if pd.isna(sub):
        continue
    label_records.append({
        'subject_id':            sub,
        'updrs_iii_walking_on':  pd.to_numeric(row[COL_ON_UPDRS_WK],  errors='coerce'),
        'updrs_iii_walking_off': pd.to_numeric(row[COL_OFF_UPDRS_WK], errors='coerce'),
        'hoehn_yahr_on':         pd.to_numeric(row[COL_ON_HY],         errors='coerce'),
        'hoehn_yahr_off':        pd.to_numeric(row[COL_OFF_HY],        errors='coerce'),
        'updrs_iii_on':          pd.to_numeric(row[COL_ON_UPDRS3],     errors='coerce'),
        'updrs_iii_off':         pd.to_numeric(row[COL_OFF_UPDRS3],    errors='coerce'),
    })

label_df = pd.DataFrame(label_records).set_index('subject_id')
assert len(label_df) == 26
label_df.reset_index().to_csv(RESULTS_TABLES / 'label_mapping.csv', index=False)
print('label_mapping.csv saved. Shape:', label_df.shape)


label_mapping.csv saved. Shape: (26, 6)


In [7]:
tm = eligible_df[['trial_key','subject_id','condition']].reset_index(drop=True)
ycol = {'on':'updrs_iii_walking_on','off':'updrs_iii_walking_off'}
tm['y'] = [label_df.loc[s, ycol[c]] for s, c in zip(tm.subject_id, tm.condition)]
tm['hoehn_yahr'] = [label_df.loc[s, 'hoehn_yahr_'+c] for s, c in zip(tm.subject_id, tm.condition)]
tm['updrs_total'] = [label_df.loc[s, 'updrs_iii_'+c] for s, c in zip(tm.subject_id, tm.condition)]
assert tm['y'].notna().all(), 'trial without walking label'
assert set(tm['y'].unique()) <= {0.0, 1.0, 2.0}
tm['y'] = tm['y'].astype(int)
assert len(tm) == 881 and set(tm.trial_key) == set(eligible_trial_keys)
tm.to_csv(METADATA_DIR / 'trial_labels.csv', index=False)
print(tm.groupby(['condition','y']).size().unstack(fill_value=0))


y            0    1    2
condition               
off        117  173  121
on         264  122   84


In [8]:
# 9. Load trial-level split manifest
TRIAL_SPLIT_MANIFEST = METADATA_DIR / "trial_split_manifest.csv"
split_df = pd.read_csv(TRIAL_SPLIT_MANIFEST)
SPLIT_MAP = dict(zip(split_df['trial_key'], split_df['split']))

# Compatibility dictionary replacing the old 5-fold splits
splits = {
    'TRAIN': split_df[split_df['split'] == 'TRAIN']['trial_key'].tolist(),
    'VALIDATION': split_df[split_df['split'] == 'VALIDATION']['trial_key'].tolist(),
    'TEST': split_df[split_df['split'] == 'TEST']['trial_key'].tolist()
}

folds = [{
    'fold_id': 'trial_split'
}]



In [9]:
CACHE = PROJECT_ROOT / 'data' / 'cache'; CACHE.mkdir(exist_ok=True)
for src in ['gsn','tbsn']:
    X = np.empty((len(tm), 751, 180), np.float32)
    for i, k in enumerate(tm.trial_key):
        X[i] = pd.read_csv(NORMALIZED_DIR / src / f'{k}.csv')[feature_names].to_numpy(np.float32)
    assert np.isfinite(X).all(), f'{src}: NaN/Inf present'
    np.save(CACHE / f'{src}_X.npy', X); print(src, X.shape)


gsn (881, 751, 180)
tbsn (881, 751, 180)


In [10]:
# Cell 11 obsolete with trial split
subject_partition_map = {}


In [11]:
# 9. LABEL/CLASS DISTRIBUTION AUDIT
print('--- LABEL/CLASS DISTRIBUTION AUDIT ---')
for split in ['TRAIN', 'VALIDATION', 'TEST']:
    split_subs = subject_stats[subject_stats['split'] == split]['subject_id']
    split_labels = label_df.loc[label_df.index.intersection(split_subs)]
    print(f'\n{split} class distributions (if discrete logic applied):')
    print(split_labels[['updrs_iii_walking_on', 'updrs_iii_walking_off']].describe().T[['count', 'mean', 'std', 'min', 'max']])
    print('Note: Class representation is preserved as far as mathematically possible given the small subject count.')


--- LABEL/CLASS DISTRIBUTION AUDIT ---

TRAIN class distributions (if discrete logic applied):
                       count      mean       std  min  max
updrs_iii_walking_on    17.0  0.647059  0.861770  0.0  2.0
updrs_iii_walking_off   16.0  1.000000  0.816497  0.0  2.0
Note: Class representation is preserved as far as mathematically possible given the small subject count.

VALIDATION class distributions (if discrete logic applied):
                       count  mean      std  min  max
updrs_iii_walking_on     4.0   0.5  0.57735  0.0  1.0
updrs_iii_walking_off    3.0   2.0  0.00000  2.0  2.0
Note: Class representation is preserved as far as mathematically possible given the small subject count.

TEST class distributions (if discrete logic applied):
                       count  mean       std  min  max
updrs_iii_walking_on     4.0  0.75  0.957427  0.0  2.0
updrs_iii_walking_off    4.0  0.75  0.500000  0.0  1.0
Note: Class representation is preserved as far as mathematically possible g

In [ ]:

import time
import numpy as np
import json
import pandas as pd
from src.preprocessing.scaling import ChannelScaler

print('Applying trial_split scaling dynamically (Optimized)...')
t0_scale = time.time()
SCALED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'scaled_features'
NORMALIZED_DIR = PROJECT_ROOT / 'data' / 'normalized'

# Pre-load normalized data to save time during loops
gsn_trials = {}
tbsn_trials = {}
for tkey in eligible_trial_keys:
    if 'gsn' in [b.split('_')[0] for b in BRANCHES]:
        gsn_trials[tkey] = pd.read_csv(NORMALIZED_DIR / 'gsn' / f'{tkey}.csv')
    if 'tbsn' in [b.split('_')[0] for b in BRANCHES]:
        tbsn_trials[tkey] = pd.read_csv(NORMALIZED_DIR / 'tbsn' / f'{tkey}.csv')

first_df = gsn_trials[eligible_trial_keys[0]]
csv_header = ','.join(first_df.columns)
csv_fmt = ['%d', '%.3f'] + ['%.8g'] * len(feature_names)

for branch in BRANCHES:
    method = branch.split('_')[1]
    norm_type = branch.split('_')[0]
    out_dir = SCALED_DIR / branch / 'trial_split'
    out_dir.mkdir(parents=True, exist_ok=True)
    
    all_trials = gsn_trials if norm_type == 'gsn' else tbsn_trials
            
    scaler = ChannelScaler(method=method)
    train_trials_dict = {k: all_trials[k] for k in splits['TRAIN'] if k in all_trials}
    all_subjects = list(set([k.split('_')[0] for k in splits['TRAIN']]))
    scaler.fit(trials=train_trials_dict, train_subjects=all_subjects, feature_names=feature_names)
    
    # Extract vectorized scaling parameters for fast np.savetxt
    scale_center = np.array([scaler.params_[f]['mean' if method == 'zscore' else 'min'] for f in feature_names], dtype=np.float64)
    scale_scale = np.array([scaler.params_[f]['std' if method == 'zscore' else 'range'] for f in feature_names], dtype=np.float64)
    
    # Transform all subjects (Train, Val, Test) and save rapidly
    for tkey in eligible_trial_keys:
        orig_df = all_trials[tkey]
        frame_time = orig_df[['frame', 'time']].to_numpy()
        raw_sig = orig_df[feature_names].to_numpy(dtype=np.float64)
        scaled_sig = (raw_sig - scale_center) / scale_scale
        full_arr = np.hstack([frame_time, scaled_sig])
        np.savetxt(out_dir / f'{tkey}.csv', full_arr, delimiter=',', header=csv_header, comments='', fmt=csv_fmt)
    
    # Save scaler parameters for downstream inference
    with open(out_dir / 'scaler_params.json', 'w') as f:
        json.dump(scaler.to_dict(), f, indent=2)
        
print(f'Finished trial_split scaling rapidly in {time.time()-t0_scale:.1f}s.')


Applying trial_split scaling dynamically (Optimized)...
Finished trial_split scaling rapidly in 260.0s.


In [ ]:

# Strict safety: Never delete source data. Only overwrite specific files if needed.
import src.preprocessing.windowing as win
t0 = time.time()
window_manifest_records = []
rejection_records = []
trial_window_counts = []  # (trial_key, branch, fold_id, n_windows)

for branch in BRANCHES:
    norm_type = branch.split('_')[0]  # 'gsn' or 'tbsn'
    for fold_info in folds:
        fold_id  = fold_info['fold_id']
        fold_str = str(fold_id)
        scaled_fold_dir = SCALED_DIR / branch / fold_str
        assert scaled_fold_dir.exists(), f'Missing: {scaled_fold_dir}'

        out_fold_dir = FINAL_DIR / branch / fold_str
        out_fold_dir.mkdir(parents=True, exist_ok=True)

        for trial_key in eligible_trial_keys:
            scaled_file = scaled_fold_dir / f'{trial_key}.csv'
            mask_file = NORMALIZED_DIR / norm_type / f'{trial_key}_mask.csv'
            assert scaled_file.exists(), f'Missing scaled: {scaled_file}'
            assert mask_file.exists(), f'Missing mask: {mask_file}'

            parts      = trial_key.split('_')
            subject_id = parts[0]
            condition  = parts[1]
            partition  = SPLIT_MAP[trial_key]

            scaled_df = pd.read_csv(scaled_file)
            mask_df   = pd.read_csv(mask_file)

            # Verify source sequence is exactly W frames
            assert len(scaled_df) == W, (
                f'{trial_key}: expected {W} frames, got {len(scaled_df)}'
            )
            assert all(f in scaled_df.columns for f in feature_names), (
                f'{trial_key}: missing feature columns in scaled CSV'
            )

            trial_meta = {
                'subject_id': subject_id,
                'trial_key': trial_key,
                'condition': condition,
                'branch': branch,
                'fold_id': fold_id,
                'partition': partition,
            }

            X_wins, M_wins, meta_list, rej_list = win.extract_sliding_windows(
                scaled_df, mask_df, feature_names, W, S, trial_meta
            )
            rejection_records.extend(rej_list)
            trial_window_counts.append((
                trial_key, branch, fold_id, len(X_wins)
            ))

            for X_w, M_w, w_meta in zip(X_wins, M_wins, meta_list):
                # Condition-matched clinical label from PDGinfo.xlsx ONLY
                if condition == 'on':
                    u_walk = label_df.loc[subject_id, 'updrs_iii_walking_on']
                    h_yahr = label_df.loc[subject_id, 'hoehn_yahr_on']
                    u_tot  = label_df.loc[subject_id, 'updrs_iii_on']
                else:
                    u_walk = label_df.loc[subject_id, 'updrs_iii_walking_off']
                    h_yahr = label_df.loc[subject_id, 'hoehn_yahr_off']
                    u_tot  = label_df.loc[subject_id, 'updrs_iii_off']

                w_meta.update({
                    'label_updrs_iii_walking': float(u_walk) if not pd.isna(u_walk) else None,
                    'label_hoehn_yahr':        float(h_yahr) if not pd.isna(h_yahr) else None,
                    'label_updrs_iii_total':   float(u_tot)  if not pd.isna(u_tot)  else None,
                    'label_missing_flag':      bool(pd.isna(u_walk)),
                })

                win_idx = w_meta['window_index']
                np.save(out_fold_dir / f'{trial_key}_win{win_idx:03d}_X.npy', X_w)
                np.save(out_fold_dir / f'{trial_key}_win{win_idx:03d}_mask.npy', M_w)
                window_manifest_records.append(w_meta)

elapsed = time.time() - t0
print(f'Windowing complete in {elapsed:.1f}s')
print(f'Windows generated : {len(window_manifest_records)}')
print(f'Rejections        : {len(rejection_records)}')



Windowing complete in 220.9s
Windows generated : 3524
Rejections        : 0


In [14]:
training_config = {'cv': 'subject-level stratified 5-fold + trial_split', 'class_weighting': 'inverse_class_frequency (train only)', 'checkpoint': 'lowest_unweighted_validation_loss', 'early_stopping_patience': 10, 'variants': {'hist': 'inherited configs, no augmentation', 'prop': 'wd=1e-2, noise 0.01 + shift 5, AdamW'}}

with open(PROJECT_ROOT / 'configs' / 'training_config.yaml', 'w') as f:
    yaml.dump(training_config, f, sort_keys=False)

print('Updated training_config.yaml')




Updated training_config.yaml


In [ ]:

window_manifest = pd.DataFrame(window_manifest_records)
window_manifest.to_csv(RESULTS_TABLES / 'window_manifest.csv', index=False)

summary = (
    window_manifest
    .groupby(['branch', 'fold_id', 'partition'])
    .size()
    .reset_index(name='window_count')
)
summary.to_csv(RESULTS_TABLES / 'windowing_summary.csv', index=False)

rej_df = pd.DataFrame(rejection_records)
if rej_df.empty:
    rej_df = pd.DataFrame(columns=[
        'subject_id', 'trial_key', 'branch', 'fold_id',
        'candidate_start_index', 'candidate_end_index', 'rejection_reason'
    ])
rej_df.to_csv(RESULTS_TABLES / 'window_rejection_audit.csv', index=False)

label_audit = (
    window_manifest
    .groupby(['branch', 'fold_id'])
    .agg(
        total_windows=('trial_key', 'count'),
        windows_with_updrs_label=('label_missing_flag', lambda x: (x == False).sum()),
        windows_missing_updrs_label=('label_missing_flag', 'sum'),
    )
    .reset_index()
)
label_audit.to_csv(RESULTS_TABLES / 'window_label_audit.csv', index=False)

print('Tables saved: window_manifest, windowing_summary, window_rejection_audit, window_label_audit')


Tables saved: window_manifest, windowing_summary, window_rejection_audit, window_label_audit


In [16]:
# 15. FINAL READY CHECK
ready = True
checks = []

has_split_file = (METADATA_DIR / 'trial_split_manifest.csv').exists()
checks.append(('Split manifest saved', has_split_file))

split_df = pd.read_csv(METADATA_DIR / 'train_val_test_split.csv')
all_in_one = (split_df['split'].value_counts().sum() == len(split_df))
checks.append(('All subjects belong to exactly one split', all_in_one))

t_subs = set()
v_subs = set()
test_subs = set()

no_overlap = True  # Trial split intentionally allows subject overlap across partitions
checks.append(('No subject leakage exists', no_overlap))

eligible_trials = sum(split_df['number_of_trials'])
manifest_trials = len(eligible_trial_keys)
checks.append(('Medication states audited', 'medication_states_present' in split_df.columns))

for desc, passed in checks:
    print(f"{desc}: {'PASS' if passed else 'FAIL'}")
    if not passed:
        ready = False

print('\nFINAL STATUS:', 'READY' if ready else 'NOT_READY')



Split manifest saved: PASS
All subjects belong to exactly one split: PASS
No subject leakage exists: PASS
Medication states audited: PASS

FINAL STATUS: READY
